# Lasmoid 100M Scratch Pretraining
### Dual-GPU DDP (Kaggle T4×2) · FineWeb-Edu + Cosmopedia · WSD Schedule · Muon/AdamW
*Pure CE loss — no distillation, no teacher model.*

This notebook pretrains the **100M parameter Lasmoid model** from scratch using the custom Lasmoid architecture (SSM + MoE + MHC + concept memory). It is optimized for maximum speed on Turing (T4) GPUs using FP16 mixed precision and the dual Muon + AdamW optimizer suite.


In [ ]:
!pip install -q uv
!uv pip install --system -q git+https://github.com/huggingface/transformers.git accelerate>=1.6.0 datasets>=3.6.0 safetensors>=0.5.3 sentencepiece einops tqdm matplotlib psutil tokenizers

import os, sys, json, time, math, random, gc, shutil, re
from pathlib import Path
from typing import Iterator, Optional
from collections import defaultdict

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
os.environ["TORCH_FORCE_SPAWN"] = "1"

import multiprocessing as mp
mp.set_start_method('spawn', force=True)

import torch
import numpy as np
import transformers
print(f"Transformers: {transformers.__version__}")
print("✅ Environment ready")


In [ ]:
import subprocess, os
from pathlib import Path

def count_gpus_safe():
    try:
        out = subprocess.check_output(["nvidia-smi", "-L"], stderr=subprocess.DEVNULL).decode("utf-8")
        gpus = [line for line in out.strip().split("\n") if line.strip()]
        if gpus:
            return len(gpus)
    except Exception:
        pass
    
    nv_gpus = Path("/proc/driver/nvidia/gpus")
    if nv_gpus.exists():
        try:
            return len(list(nv_gpus.iterdir()))
        except Exception:
            pass
            
    cvd = os.environ.get("CUDA_VISIBLE_DEVICES")
    if cvd is not None:
        if cvd == "":
            return 0
        return len(cvd.split(","))
        
    return 0

NUM_GPUS = count_gpus_safe()
print(f"Detected {NUM_GPUS} GPUs (without initializing PyTorch CUDA)")

if NUM_GPUS == 0:
    raise RuntimeError("No GPU found. Enable T4 x2 in Kaggle Settings → Accelerator.")

# Print GPU info via nvidia-smi subprocess
try:
    print(subprocess.check_output(["nvidia-smi"]).decode("utf-8"))
except Exception:
    print("nvidia-smi not available or failed.")

DTYPE = torch.float32 
print(f"Model initialization dtype: {DTYPE}")


In [ ]:
# ── Repo & paths ────────────────────────────────────────────────────────────
REPO_URL   = "https://github.com/Theory903/Lasmoid.git"
REPO_DIR   = Path("/kaggle/working/Lasmoid")
CKPT_DIR   = Path("/kaggle/working/checkpoints/lasmoid_scratch_100m")
LOG_DIR    = Path("/kaggle/working/logs/lasmoid_scratch_100m")
EXPORT_DIR = Path("/kaggle/working/export/lasmoid_scratch_100m")

for d in [CKPT_DIR, LOG_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Clone/pull repo
if not (REPO_DIR / "inference").exists():
    os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull --rebase --autostash")

# Add to sys.path so we can import Lasmoid modules
for d in [str(REPO_DIR), str(REPO_DIR / "inference"), str(REPO_DIR / "train")]:
    if d not in sys.path:
        sys.path.insert(0, str(d))

print(f"Repo: {REPO_DIR}")
print(f"CKPT: {CKPT_DIR}")
print(f"Logs: {LOG_DIR}")
print("✅ Sys.path ready")


In [ ]:
from transformers import AutoTokenizer

print("Loading default tokenizer from the repository root...")
tokenizer = AutoTokenizer.from_pretrained(str(REPO_DIR), use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE = len(tokenizer)

print(f"✅ Tokenizer loaded: vocab_size={VOCAB_SIZE:,}")
print(f"   BOS={tokenizer.bos_token_id} EOS={tokenizer.eos_token_id} PAD={tokenizer.pad_token_id}")


In [ ]:
# 100M parameter model config matched to configs/model/config_gemma4_100m.json
LASMOID_CONFIG = {
    "vocab_size": VOCAB_SIZE,       # 262144
    "max_seq_len": 512,
    "max_batch_size": 4,
    "dtype": "fp16",
    "norm_eps": 1e-06,
    "rope_theta": 10000.0,
    "rope_factor": 1.0,
    "beta_fast": 32,
    "beta_slow": 1,
    "original_seq_len": 0,
    "swiglu_limit": 10.0,
    "num_residual_streams": 4,
    "hc_sinkhorn_iters": 8,
    "hc_eps": 1e-06,
    "hcm_ema_alpha": 0.99,
    "hcm_commit_loss_coeff": 0.25,
    "entropy_threshold": 0.5,
    "router_z_loss_coeff": 0.001,
    "ema_bias_lr": 0.01,
    "expert_capacity_factor": 1.25,
    "moe_load_balance_coeff": 0.01,
    "predictive_coding_coeff": 0.01,
    "reasoning_steps": 2,
    "think_token_id": 107,
    "answer_token_id": 108,
    "cot_exit_confidence": 0.9,
    "moe_router_entropy_coeff": 0.001,
    "moe_capacity_loss_coeff": 0.01,
    "token_concept_loss_coeff": 0.05,
    "steering_attributes": [
        "creativity",
        "helpfulness",
        "complexity",
        "scientific_rigor"
    ],
    "post_attn_norm": True,
    "post_ffw_norm": True,
    "moe_dual_ffn": True,
    "ssm_d_skip": True,
    # 100M architectural scaling parameters
    "dim": 256,
    "n_layers": 12,
    "n_heads": 4,
    "q_lora_rank": 64,
    "head_dim": 48,
    "rope_head_dim": 16,
    "o_groups": 2,
    "o_lora_rank": 64,
    "n_routed_experts": 8,
    "n_shared_experts": 1,
    "n_activated_experts": 2,
    "moe_latent_dim": 128,
    "num_concepts": 64,
    "num_abstract_concepts": 8,
    "num_global_concepts": 2,
    "codebook_size": 256,
    "lightning_topk_blocks": 2,
    "ssm_heads": 4,
    "ssm_state_dim": 16,
    "ssm_kernel_size": 4,
    "ssm_chunk_size": 64,
    "ssm_dt_min": 0.001,
    "ssm_dt_max": 0.1,
    "ssm_dt_init_floor": 0.0001,
    "ssm_n_groups": 1
}

# Save config so model components can load it
config_path = EXPORT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(LASMOID_CONFIG, f, indent=2)
print(f"✅ Config saved to {config_path}")


In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
SEQ_LEN      = 512
BATCH_SIZE   = 4          # Batch size per GPU (optimizes T4 GPU core occupancy)
MAX_STEPS    = 15000
CKPT_EVERY   = 500
LOG_EVERY    = 10
VAL_EVERY    = 500
SEED         = 42

# We calculate GRAD_ACCUM dynamically to guarantee exactly 8,192 tokens/step
# NUM_GPUS is already detected in Cell 3 without CUDA initialization.
if "NUM_GPUS" not in globals():
    try:
        NUM_GPUS = count_gpus_safe()
    except NameError:
        NUM_GPUS = 1

GRAD_ACCUM   = max(1, 8192 // (BATCH_SIZE * SEQ_LEN * NUM_GPUS))

# ── Hugging Face Upload Configuration ───────────────────────────────────────
# Enter your repository ID (username/repo) to push checkpoints.
# Set a secret named "HF_TOKEN" under Add-ons -> Secrets in the Kaggle UI.
HF_REPO      = "Theory903/lasmoid-100m-scratch" 

print(f"Seq len: {SEQ_LEN}, Batch: {BATCH_SIZE}×{NUM_GPUS}GPUs×{GRAD_ACCUM}accum = {BATCH_SIZE*NUM_GPUS*GRAD_ACCUM} eff")


In [ ]:
MIN_CHARS = 200
MAX_CHARS = 8000

_BOILERPLATE = [
    "Click here", "Subscribe to", "Cookie Policy",
    "Terms of Service", "©", "All rights reserved", "Skip to content",
]

def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    for pat in _BOILERPLATE:
        if pat.lower() in text.lower():
            return ""
    return text

def stream_packed(dataset, tokenizer, seq_len: int,
                  max_batches: int = None) -> Iterator:
    """Yields (input_ids [seq_len], labels [seq_len]) via doc-packing."""
    buf = []
    count = 0
    eos_id = tokenizer.eos_token_id
    for example in dataset:
        text = clean_text(example.get("text", ""))
        if len(text) < MIN_CHARS or len(text) > MAX_CHARS:
            continue
        ids = tokenizer.encode(text, add_special_tokens=True)
        if not isinstance(ids, list):
            ids = ids.tolist() if hasattr(ids, "tolist") else list(ids)
        ids.append(eos_id)
        buf.extend(ids)
        while len(buf) >= seq_len + 1:
            chunk = buf[:seq_len + 1]
            buf = buf[seq_len + 1:]
            x = torch.tensor(chunk[:-1], dtype=torch.long)
            y = torch.tensor(chunk[1:], dtype=torch.long)
            yield x, y
            count += 1
            if max_batches and count >= max_batches:
                return

class ExpertMonitor:
    def __init__(self, n_experts: int, n_layers: int):
        self.n_experts = n_experts
        self.n_layers = n_layers
        self.reset()
    def reset(self):
        self.counts = torch.zeros(self.n_layers, self.n_experts)
        self.steps = 0
    def update(self, routing_maps: list):
        for layer_idx, routing in enumerate(routing_maps):
            if routing is None or layer_idx >= self.n_layers:
                continue
            try:
                r = routing.detach().cpu().view(-1)
                for idx in r.tolist():
                    if 0 <= int(idx) < self.n_experts:
                        self.counts[layer_idx, int(idx)] += 1
            except Exception:
                pass
        self.steps += 1
    def stats(self) -> dict:
        total = self.counts.sum(-1, keepdim=True).clamp(min=1)
        freq = self.counts / total
        dead_mask = freq < 0.01
        ent = -(freq.clamp(min=1e-8) * freq.clamp(min=1e-8).log()).sum(-1)
        return {
            "dead_experts": int(dead_mask.sum().item()),
            "dead_pct": 100 * dead_mask.sum().item() / max(1, self.n_layers * self.n_experts),
            "entropy_ratio": (ent.mean() / math.log(self.n_experts)).item() if self.n_experts > 1 else 1.0,
        }

class ConceptMonitor:
    def __init__(self, num_concepts: int):
        self.num_concepts = num_concepts
        self.reset()
    def reset(self):
        self.usage = torch.zeros(self.num_concepts)
        self.steps = 0
    def update(self, concept_indices: list):
        for ci in concept_indices:
            if ci is None:
                continue
            try:
                idxs = ci.detach().cpu().view(-1)
                for i in idxs.tolist():
                    if 0 <= int(i) < self.num_concepts:
                        self.usage[int(i)] += 1
            except Exception:
                pass
        self.steps += 1
    def stats(self) -> dict:
        total = self.usage.sum().clamp(min=1)
        freq = self.usage / total
        top5 = freq.topk(min(5, self.num_concepts)).values.sum().item()
        return {
            "collapsed": top5 > 0.80,
            "top5_coverage": top5,
            "dead_concepts": int((freq < 0.001).sum().item()),
        }

CURSOR_FILE = CKPT_DIR / "cursor.json"

def save_cursor(step: int, dataset_idx: int):
    with open(CURSOR_FILE, "w") as f:
        json.dump({"step": step, "dataset_idx": dataset_idx, "timestamp": time.time()}, f)

def load_cursor() -> dict:
    if CURSOR_FILE.exists():
        with open(CURSOR_FILE) as f:
            return json.load(f)
    return {"step": 0, "dataset_idx": 0}

def find_latest():
    latest = CKPT_DIR / "latest"
    if (latest.is_symlink() or latest.exists()) and latest.resolve().exists():
        return latest.resolve()
    c = sorted(CKPT_DIR.glob("step_*"))
    return c[-1] if c else None


In [ ]:
def train_fn():
    import os, sys, json, time, math, random, gc, shutil, re, signal
    from pathlib import Path
    import torch
    from torch.nn import functional as F
    from accelerate import Accelerator, DistributedDataParallelKwargs
    from safetensors.torch import save_file as sf_save, load_file as sf_load
    from tqdm.auto import tqdm
    
    # ── Accelerator Setup ───────────────────────────────────────────────────
    ddp_kwargs = DistributedDataParallelKwargs(find_unused_parameters=False)
    accelerator = Accelerator(
        mixed_precision="fp16",       # Turing T4 native mixed precision
        gradient_accumulation_steps=GRAD_ACCUM,
        kwargs_handlers=[ddp_kwargs],
    )
    
    DEVICE  = accelerator.device
    IS_MAIN = accelerator.is_main_process
    N_PROC  = accelerator.num_processes
    
    random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        
    # ── Setup Model ─────────────────────────────────────────────────────────
    from inference.config import ModelArgs
    from inference.lasmoid import Lasmoid
    
    model_args = ModelArgs(**LASMOID_CONFIG)
    # Instantiate in float32 for mixed precision stability!
    model = Lasmoid(model_args).to(torch.float32)
    model.gradient_checkpointing = True
    
    total_params = sum(p.numel() for p in model.parameters())
    
    # ── Optimizers & Scheduler ──────────────────────────────────────────────
    from train.optimizer import (
        build_optimizers,
        ensure_muon_closure_compat,
    )
    from train.scheduler import WSDScheduler
    
    ensure_muon_closure_compat()
    optimizers = build_optimizers(
        model,
        muon_lr=2e-3,
        adamw_lr=3e-4,
        weight_decay=0.1,
        betas=(0.9, 0.95),
        eps=1e-8,
        momentum=0.95,
        ns_steps=5,
        adaptive_noise=False,
    )
    
    WARMUP_STEPS = max(1, int(0.02 * MAX_STEPS))
    STABLE_STEPS = int(0.80 * MAX_STEPS)
    DECAY_STEPS  = MAX_STEPS - WARMUP_STEPS - STABLE_STEPS
    
    base_lrs = [[g["lr"] for g in opt.param_groups] for opt in optimizers]
    
    scheduler = WSDScheduler(
        optimizers=optimizers,
        warmup_steps=WARMUP_STEPS,
        stable_steps=STABLE_STEPS,
        decay_steps=DECAY_STEPS,
        base_lrs=base_lrs,
        min_lr_ratio=0.1,
    )
    
    model, *optimizers = accelerator.prepare(model, *optimizers)
    _unwrapped = accelerator.unwrap_model(model)
    
    # ── Hugging Face Uploader ───────────────────────────────────────────────
    hf_uploader = None
    if IS_MAIN and HF_REPO:
        try:
            from hf_uploader import HFAnyUploader
            hf_uploader = HFAnyUploader(repo_id=HF_REPO)
        except Exception as e:
            if IS_MAIN:
                print(f"⚠️ Failed to init Hugging Face Uploader: {e}")
                
    # ── Monitors ────────────────────────────────────────────────────────────
    expert_monitor = ExpertMonitor(
        n_experts=model_args.n_routed_experts, n_layers=model_args.n_layers
    )
    concept_monitor = ConceptMonitor(num_concepts=model_args.num_concepts)
    
    # ── Checkpoint System ───────────────────────────────────────────────────
    def save_checkpoint(step, dataset_idx, loss_history, expert_stats, concept_stats):
        if not IS_MAIN:
            return
        ckpt_path = CKPT_DIR / f"step_{step:06d}"
        ckpt_path.mkdir(parents=True, exist_ok=True)
        
        # Save weights
        sd = _unwrapped.state_dict()
        sf_save({k: v.cpu() for k, v in sd.items()}, str(ckpt_path / "model.safetensors"))
        
        # Save optimizers
        opt_states = [opt.state_dict() for opt in optimizers]
        torch.save(opt_states, ckpt_path / "optimizer.pt")
        
        # Save RNG
        torch.save({
            "cpu": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "python": random.getstate(),
        }, ckpt_path / "rng.pt")
        
        # Save Meta
        with open(ckpt_path / "meta.json", "w") as f:
            json.dump({
                "step": step,
                "dataset_idx": dataset_idx,
                "loss_history": loss_history[-500:],
                "expert_stats": expert_stats,
                "concept_stats": concept_stats,
                "timestamp": time.time(),
            }, f)
            
        save_cursor(step, dataset_idx)
        latest = CKPT_DIR / "latest"
        if latest.is_symlink() or latest.exists():
            latest.unlink(missing_ok=True)
        try:
            latest.symlink_to(ckpt_path.name)
        except (OSError, AttributeError):
            pass
        print(f"  💾 CKPT: step {step} → {ckpt_path.name}")
        
        if hf_uploader:
            try:
                shutil.copy(str(EXPORT_DIR / "config.json"), str(ckpt_path / "config.json"))
                for fname in ["tokenizer.json", "tokenizer_config.json"]:
                    src = REPO_DIR / fname
                    if src.exists():
                        shutil.copy(str(src), str(ckpt_path / fname))
            except Exception as e:
                print(f"  ⚠️ Failed to copy config/tokenizer for HF: {e}")
            hf_uploader.upload_folder_async(
                folder_path=ckpt_path,
                path_in_repo=f"checkpoints/step_{step:06d}",
                commit_message=f"Checkpoint at step {step}"
            )
            
    # ── Shutdown handler ────────────────────────────────────────────────────
    _CHECKPOINT_ON_KILL = {"step": 0, "dataset_idx": 0}
    def _shutdown_handler(signum, frame):
        sig_name = signal.Signals(signum).name
        print(f"\n⚠️  {sig_name} — saving emergency checkpoint...")
        step = _CHECKPOINT_ON_KILL.get("step", 0)
        ds_idx = _CHECKPOINT_ON_KILL.get("dataset_idx", 0)
        if step > 0:
            ed = CKPT_DIR / "emergency"
            ed.mkdir(parents=True, exist_ok=True)
            sf_save({k: v.cpu() for k, v in _unwrapped.state_dict().items()},
                    str(ed / "model.safetensors"))
            save_cursor(step, ds_idx)
            print(f"  💾 Emergency saved at step {step}")
            if hf_uploader:
                try:
                    shutil.copy(str(EXPORT_DIR / "config.json"), str(ed / "config.json"))
                    for fname in ["tokenizer.json", "tokenizer_config.json"]:
                        src = REPO_DIR / fname
                        if src.exists():
                            shutil.copy(str(src), str(ed / fname))
                except Exception as e:
                    pass
                hf_uploader.upload_folder_async(
                    folder_path=ed,
                    path_in_repo="checkpoints/emergency",
                    commit_message="Emergency checkpoint"
                )
                hf_uploader.wait_for_uploads()
        print("Exiting.")
        exit(0)
        
    if IS_MAIN:
        signal.signal(signal.SIGTERM, _shutdown_handler)
        signal.signal(signal.SIGINT, _shutdown_handler)
        
    # ── Validation ──────────────────────────────────────────────────────────
    from inference.loss import compute_loss
    
    def run_validation(val_steps: int = 64) -> float | None:
        _unwrapped.eval()
        try:
            from datasets import load_dataset
            val_ds = load_dataset(
                "roneneldan/TinyStories", split="validation",
                streaming=True, trust_remote_code=True,
            ).select_columns(["text"])
            
            total_nll, total_tokens, count = 0.0, 0, 0
            with torch.no_grad():
                for example in val_ds:
                    if count >= val_steps:
                        break
                    text = example.get("text", "")
                    if len(text) < 100:
                        continue
                    ids = tokenizer.encode(text, add_special_tokens=True)[:SEQ_LEN+1]
                    if len(ids) < SEQ_LEN + 1:
                        continue
                    x = torch.tensor(ids[:SEQ_LEN], dtype=torch.long).unsqueeze(0).to(DEVICE)
                    y = torch.tensor(ids[1:SEQ_LEN+1], dtype=torch.long).unsqueeze(0).to(DEVICE)
                    
                    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                        out = _unwrapped(x, x)
                        logits = out[0]
                        nll = F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                                              y.reshape(-1), reduction="sum")
                    total_nll += nll.item()
                    total_tokens += y.numel()
                    count += 1
            ppl = math.exp(min(total_nll / max(total_tokens, 1), 20))
            if IS_MAIN:
                print(f"  📊 Val PPL: {ppl:.2f} (n={count})")
            return ppl
        except Exception as e:
            if IS_MAIN:
                print(f"  ⚠️  Val error: {e}")
            return None
        finally:
            _unwrapped.train()
            
    # ── Dataset Loading & Sharding ──────────────────────────────────────────
    from datasets import load_dataset, interleave_datasets
    
    ds_fw = load_dataset(
        "HuggingFaceTB/smollm-corpus", "fineweb-edu-dedup",
        split="train", streaming=True, trust_remote_code=True,
    ).select_columns(["text"])
    
    ds_c = load_dataset(
        "HuggingFaceTB/smollm-corpus", "cosmopedia-v2",
        split="train", streaming=True, trust_remote_code=True,
    ).select_columns(["text"])
    
    ds_tr = interleave_datasets(
        [ds_fw, ds_c], probabilities=[0.70, 0.30], seed=SEED,
    )
    if N_PROC > 1:
        ds_tr = ds_tr.shard(num_shards=N_PROC, index=accelerator.process_index)
        
    data_iter = stream_packed(ds_tr, tokenizer, SEQ_LEN)
    
    # ── Resume state ────────────────────────────────────────────────────────
    loss_history = []
    start_step = 0
    dataset_idx = 0
    
    cursor = load_cursor()
    if cursor.get("step", 0) > 0:
        latest_ckpt = find_latest()
        if latest_ckpt is not None:
            if IS_MAIN:
                print(f"Resuming from {latest_ckpt}...")
            state = sf_load(str(latest_ckpt / "model.safetensors"), device="cpu")
            _unwrapped.load_state_dict(state, strict=True)
            if (latest_ckpt / "optimizer.pt").exists():
                opt_states = torch.load(latest_ckpt / "optimizer.pt", map_location="cpu")
                for opt, st in zip(optimizers, opt_states):
                    try:
                        opt.load_state_dict(st)
                    except Exception as e:
                        if IS_MAIN:
                            print(f"  ⚠️  Opt load: {e}")
            if (latest_ckpt / "meta.json").exists():
                with open(latest_ckpt / "meta.json") as f:
                    meta = json.load(f)
                start_step = meta.get("step", 0)
                loss_history = meta.get("loss_history", [])
                dataset_idx = meta.get("dataset_idx", 0)
            if IS_MAIN:
                print(f"✅ Resumed step {start_step}")
                
    # Broadcast start_step and dataset_idx
    if N_PROC > 1:
        t = torch.tensor([start_step, dataset_idx], dtype=torch.long, device=DEVICE)
        torch.distributed.broadcast(t, src=0)
        start_step = t[0].item()
        dataset_idx = t[1].item()
        
    # Fast-forward dataset
    if dataset_idx > 0:
        if IS_MAIN:
            print(f"Fast-forwarding dataset to position {dataset_idx}...")
        for _ in range(dataset_idx):
            try:
                next(data_iter)
            except StopIteration:
                data_iter = stream_packed(ds_tr, tokenizer, SEQ_LEN)
                
    # ── Logging ──────────────────────────────────────────────────────────────
    loss_log_path = LOG_DIR / "loss.jsonl"
    def log_step(d: dict):
        if not IS_MAIN:
            return
        with open(loss_log_path, "a") as f:
            f.write(json.dumps(d) + "\n")
            
    tokens_per_step = BATCH_SIZE * SEQ_LEN * GRAD_ACCUM * N_PROC
    if IS_MAIN:
        print(f"Tokens/step: {tokens_per_step:,}")
        print(f"Total       : {tokens_per_step*MAX_STEPS:,} ({tokens_per_step*MAX_STEPS/1e9:.2f}B)")
        
    # ── OOM Safety ──────────────────────────────────────────────────────────
    def safe_forward(m, x_enc, x_dec):
        try:
            return m(x_enc, x_dec)
        except torch.cuda.OutOfMemoryError:
            print("  ⚠️  OOM — skipping batch")
            torch.cuda.empty_cache(); gc.collect()
            return None
        except ValueError as e:
            if any(kw in str(e) for kw in ("stoi", "storage", "symbolize")):
                print(f"  ⚠️  Symbolizer err: {e}")
                return None
            raise

    # ── Training Loop ────────────────────────────────────────────────────────
    pbar = tqdm(range(start_step, MAX_STEPS), initial=start_step,
                total=MAX_STEPS, desc="Lasmoid 100M Scratch", disable=not IS_MAIN)
                
    for step in pbar:
        t0 = time.time()
        _CHECKPOINT_ON_KILL.update({"step": step, "dataset_idx": dataset_idx})
        lr_mult = scheduler.step(step)
        
        if hasattr(_unwrapped, "clear_saved_checkpoint_states"):
            _unwrapped.clear_saved_checkpoint_states()
            
        total_loss_accum = 0.0
        for micro in range(GRAD_ACCUM):
            try:
                x, y = next(data_iter)
                dataset_idx += 1
            except StopIteration:
                data_iter = stream_packed(ds_tr, tokenizer, SEQ_LEN)
                x, y = next(data_iter)
                
            x = x.unsqueeze(0).to(DEVICE)
            y = y.unsqueeze(0).to(DEVICE)
            
            with accelerator.accumulate(model):
                with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                    out = safe_forward(model, x, x)
                    if out is None:
                        continue
                    (logits, mtp_logits, _, _, routing_maps, concept_indices, adjacencies, event_probs) = out
                    
                    total_loss = compute_loss(
                        logits=logits,
                        targets=y,
                        routing_maps=routing_maps,
                        vq_losses=[_unwrapped.last_vq_loss],
                        adjacencies=adjacencies,
                        event_probs=event_probs,
                        moe_aux_loss=_unwrapped.last_moe_loss,
                        moe_aux_coeff=1.0,
                        mtp_loss=(F.cross_entropy(
                            mtp_logits[:, :y.shape[1]-1].reshape(-1, mtp_logits.shape[-1]),
                            y[:, 1:].reshape(-1), ignore_index=-100,
                        ) if mtp_logits is not None and x.shape[1] > 1 else None),
                        mtp_coeff=0.3,
                        token_concept_loss=_unwrapped.last_token_concept_loss,
                        token_concept_coeff=0.05,
                        commit_loss=_unwrapped.last_commit_loss,
                        commit_coeff=0.25,
                        curiosity_loss=None,
                        graph_sparsity=0.01,
                        label_smoothing=0.0,
                        ignore_index=-100,
                    )
                    
                accelerator.backward(total_loss / GRAD_ACCUM)
                total_loss_accum += total_loss.item() / GRAD_ACCUM
                
            expert_monitor.update(routing_maps)
            concept_monitor.update(concept_indices)
            
        # Gradient clipping and optimizer step
        if accelerator.sync_gradients:
            grad_norm = accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            _skip = False
            if math.isnan(grad_norm) or math.isinf(grad_norm):
                for _p in model.parameters():
                    if _p.grad is not None and (torch.isnan(_p.grad).any() or torch.isinf(_p.grad).any()):
                        _skip = True
                        break
            if _skip:
                if IS_MAIN:
                    print(f"  ⚠️  NaN/Inf grad at step {step} — skipping")
                log_step({"step": step, "nan_gradient": True, "grad_norm": float("nan")})
                for opt in optimizers:
                    opt.zero_grad(set_to_none=True)
            else:
                for opt in optimizers:
                    opt.step()
                _unwrapped.apply_pending_bias_updates()
                for opt in optimizers:
                    opt.zero_grad(set_to_none=True)
                    
        loss_history.append(total_loss_accum)
        step_time = time.time() - t0
        
        if IS_MAIN:
            exp_s = expert_monitor.stats()
            con_s = concept_monitor.stats()
            tokens_per_sec = tokens_per_step / max(step_time, 1e-6)
            
            if torch.cuda.is_available() and step % 10 == 0:
                _alloc = torch.cuda.memory_allocated() / 1024**3
                _total = torch.cuda.get_device_properties(0).total_memory / 1e9
                _mem_str = f"{_alloc:.1f}G/{_total-_alloc:.1f}G free"
            else:
                _mem_str = ""
                
            pbar.set_postfix({
                "loss": f"{total_loss_accum:.3f}",
                "tok/s": f"{tokens_per_sec:,.0f}",
                "mem": _mem_str,
                "ent": f"{exp_s['entropy_ratio']:.2f}",
                "dead": exp_s["dead_experts"],
            })
            
            if step % LOG_EVERY == 0:
                log_step({
                    "step": step,
                    "loss": total_loss_accum,
                    "step_time_s": step_time,
                    "tokens_per_sec": tokens_per_sec,
                    "lr_mult": lr_mult,
                    "grad_norm": grad_norm if accelerator.sync_gradients else 0.0,
                    "gpu_mem_gb": _alloc if torch.cuda.is_available() else 0,
                    "expert_stats": exp_s,
                    "concept_stats": con_s,
                })
                
            # Checkpoint saving
            if step > 0 and step % CKPT_EVERY == 0:
                save_checkpoint(step, dataset_idx, loss_history, exp_s, con_s)
                expert_monitor.reset()
                concept_monitor.reset()
                
            # Validation
            if step > 0 and step % VAL_EVERY == 0:
                val_ppl = run_validation(val_steps=64)
                if val_ppl is not None:
                    log_step({"step": step, "val_ppl": val_ppl})
                    
    # Save final
    if IS_MAIN:
        print("\n✅ Training complete!")
        save_checkpoint(MAX_STEPS, dataset_idx, loss_history,
                        expert_monitor.stats(), concept_monitor.stats())
        final_ppl = run_validation(val_steps=128)
        if final_ppl:
            print(f"🎯 Final val PPL: {final_ppl:.2f}")
            
        print(f"Exporting final model to {EXPORT_DIR}...")
        sf_save(
            {k: v.cpu() for k, v in _unwrapped.state_dict().items()},
            str(EXPORT_DIR / "model.safetensors"),
        )
        shutil.copy(str(config_path), EXPORT_DIR / "config.json")
        tokenizer.save_pretrained(str(EXPORT_DIR / "tokenizer"))
        
        # Save training summary
        summary = {
            "model": "Lasmoid",
            "params_m": total_params / 1e6,
            "config": LASMOID_CONFIG,
            "training": {
                "max_steps": MAX_STEPS,
                "batch_size": BATCH_SIZE,
                "grad_accum": GRAD_ACCUM,
                "seq_len": SEQ_LEN,
                "n_gpu": N_PROC,
                "total_tokens": tokens_per_step * MAX_STEPS,
                "muon_lr": 2e-3,
                "adamw_lr": 3e-4,
                "scheduler": "WSD",
            },
            "final_loss": loss_history[-1] if loss_history else None,
        }
        with open(EXPORT_DIR / "summary.json", "w") as f:
            json.dump(summary, f, indent=2, default=str)
            
        if hf_uploader:
            hf_uploader.upload_folder_async(
                folder_path=EXPORT_DIR,
                path_in_repo="final_model",
                commit_message="Final trained model export"
            )
            hf_uploader.wait_for_uploads()
            print("🎉 Success! All checkpoints and final model uploaded to Hugging Face.")

# ── Launch Distributed Training ───────────────────────────────────────────────
if NUM_GPUS > 1:
    from torch.distributed.launcher.api import LaunchConfig, elastic_launch
    config = LaunchConfig(
        min_nodes=1,
        max_nodes=1,
        nproc_per_node=NUM_GPUS,
        run_id="notebook",
        role="notebook",
        rdzv_endpoint="127.0.0.1:0",
        rdzv_backend="static",
        max_restarts=3,
        monitor_interval=5,
        start_method="spawn",
    )
    print(f"Launching distributed training on {NUM_GPUS} GPUs (spawn)...")
    elastic_launch(config=config, entrypoint=train_fn)()
else:
    print("Launching single-process training...")
    train_fn()


In [ ]:
loss_log_path = LOG_DIR / "loss.jsonl"
loss_history = []
if loss_log_path.exists():
    with open(loss_log_path, "r") as f:
        for line in f:
            try:
                data = json.loads(line)
                if "loss" in data:
                    loss_history.append(data["loss"])
            except Exception:
                pass

if loss_history:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 1, figsize=(12, 4))
    ax.plot(loss_history, alpha=0.6, label="train loss")
    if len(loss_history) > 50:
        window = 50
        smooth = np.convolve(loss_history, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(loss_history)), smooth,
                color='red', linewidth=2, label=f"{window}-step avg")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title(f"Lasmoid 100M Scratch Pretraining")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(LOG_DIR / "training_loss.png"), dpi=150, bbox_inches="tight")
    print(f"📊 Plot saved: {LOG_DIR / 'training_loss.png'}")
else:
    print("No loss history found to plot.")
